<a href="https://colab.research.google.com/github/marifinilham-ds/east-java-positive-deviance-few/blob/main/notebooks/01_earth_engine_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='positive-deviance-east-java')
print("Earth Engine OK")

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=vC32cvOJecZuGDm0EyIxzMLV40n5RILjTlDoQLwcRzA&tc=POszNlGEz99PoSgx-5xqlQGJvk_oO-xYB4l7gM8rqJY&cc=s9W9vr2g6dRb2RPNv_zFBN7821uLAq_fS9Jr2xXeFls

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AXlqoi5Lua5tuawkSc_u7LORswCZRlfA3Dv3KW_ZuVL2FxhEyMi6hCERFAg

Successfully saved authorization token.
Earth Engine OK


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
gaul = ee.FeatureCollection('FAO/GAUL/2015/level2')
indo = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia'))
jatim = indo.filter(ee.Filter.eq('ADM1_NAME', 'Jawa Timur'))
print(jatim.size().getInfo())

/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


39


In [ ]:
# 1) Daftar nama dari versi lama (39 unit), untuk melihat unit ekstranya
old = ee.FeatureCollection('FAO/GAUL/2015/level2') \
        .filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')) \
        .filter(ee.Filter.eq('ADM1_NAME', 'Jawa Timur'))
print(sorted(old.aggregate_array('ADM2_NAME').getInfo()))

# 2) Skema versi 2025 (nama kolom)
new = ee.FeatureCollection('FAO/GAUL/2025/level2')
print(new.first().propertyNames().getInfo())

['Bangkalan', 'Banyuwangi', 'Blitar', 'Bojonegoro', 'Bondowoso', 'Gresik', 'Jember', 'Jombang', 'Kediri', 'Kota Batu', 'Kota Blitar', 'Kota Jember', 'Kota Kediri', 'Kota Madiun', 'Kota Malang', 'Kota Mojokerto', 'Kota Pasuruan', 'Kota Probolinggo', 'Kota Surabaya', 'Lamongan', 'Lumajang', 'Madiun', 'Magetan', 'Malang', 'Mojokerto', 'Nganjuk', 'Ngawi', 'Pacitan', 'Pamekasan', 'Pasuruan', 'Ponorogo', 'Probolinggo', 'Sampang', 'Sidoarjo', 'Situbondo', 'Sumenep', 'Trenggalek', 'Tuban', 'Tulungagung']
['FID', 'GAUL0_CODE', 'GAUL1_NAME', 'GAUL2_CODE', 'GAUL2_NAME', 'DISP_EN', 'GAUL0_NAME', 'CONTINENT', 'ISO3_CODE', 'MAP_CODE', 'system:index', 'GAUL1_CODE']


In [ ]:
new = ee.FeatureCollection('FAO/GAUL/2025/level2') \
        .filter(ee.Filter.eq('GAUL0_NAME', 'Indonesia')) \
        .filter(ee.Filter.eq('GAUL1_NAME', 'Jawa Timur'))
print('jumlah 2025:', new.size().getInfo())

names_new = sorted(new.aggregate_array('GAUL2_NAME').getInfo())
names_old = set(old.aggregate_array('ADM2_NAME').getInfo())
print('hanya di 2025:', sorted(set(names_new) - names_old))
print('hanya di 2015:', sorted(names_old - set(names_new)))

# Luas Jember vs Kota Jember (versi 2015)
for n in ['Jember', 'Kota Jember']:
    g = old.filter(ee.Filter.eq('ADM2_NAME', n)).geometry()
    print(n, round(g.area(1).divide(1e6).getInfo(), 1), 'km2')

jumlah 2025: 38
hanya di 2025: []
hanya di 2015: ['Kota Jember']
Jember 3234.4 km2
Kota Jember 99.3 km2


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/marifinilham-ds/east-java-positive-deviance-few/main/data/raw/bps/padi_jatim_2019_2022.csv"
padi = pd.read_csv(url)

bps = padi[['kabupaten_kota', 'jenis']].drop_duplicates().copy()
bps['gaul_name'] = bps.apply(
    lambda r: 'Kota ' + r.kabupaten_kota if r.jenis == 'kota' else r.kabupaten_kota,
    axis=1)

print('unit BPS:', len(bps))
print('di BPS, tidak ada di GAUL:', sorted(set(bps.gaul_name) - set(names_new)))
print('di GAUL, tidak ada di BPS:', sorted(set(names_new) - set(bps.gaul_name)))

unit BPS: 38
di BPS, tidak ada di GAUL: []
di GAUL, tidak ada di BPS: []


In [ ]:
import pandas as pd

regions = new.select(['GAUL2_NAME'])
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')
modis = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')

def to_df(fc):
    feats = fc.getInfo()['features']
    return pd.DataFrame([f['properties'] for f in feats])

frames = []
for y in [2019, 2020, 2021, 2022]:
    # Curah hujan: total tahunan dan musim kemarau (Jun-Sep), satuan mm
    rain_ann = chirps.filterDate(f'{y}-01-01', f'{y+1}-01-01').sum().rename('rain_annual_mm')
    rain_dry = chirps.filterDate(f'{y}-06-01', f'{y}-10-01').sum().rename('rain_dry_mm')
    rain_img = rain_ann.addBands(rain_dry)
    rain_fc = rain_img.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=5566)
    rain_df = to_df(rain_fc)

    # NDVI: rata-rata dan maksimum tahunan (skala asli 0.0001)
    nd = modis.filterDate(f'{y}-01-01', f'{y+1}-01-01')
    ndvi_img = (nd.mean().multiply(0.0001).rename('ndvi_mean')
                .addBands(nd.max().multiply(0.0001).rename('ndvi_max')))
    ndvi_fc = ndvi_img.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=250)
    ndvi_df = to_df(ndvi_fc)

    df = rain_df.merge(ndvi_df, on='GAUL2_NAME')
    df['tahun'] = y
    frames.append(df)
    print('selesai', y)

sat = pd.concat(frames, ignore_index=True).rename(columns={'GAUL2_NAME': 'gaul_name'})
print(sat.shape)
print(sat.isna().sum())
print(sat.groupby('tahun')[['rain_annual_mm', 'rain_dry_mm', 'ndvi_mean', 'ndvi_max']].describe().T.round(2))

selesai 2019
selesai 2020
selesai 2021
selesai 2022
(152, 6)
gaul_name         0
rain_annual_mm    0
rain_dry_mm       0
ndvi_max          0
ndvi_mean         0
tahun             0
dtype: int64
tahun                    2019     2020     2021     2022
rain_annual_mm count    38.00    38.00    38.00    38.00
               mean   1620.12  2333.89  2533.44  2688.62
               std     213.03   249.62   382.40   379.79
               min    1226.35  1753.12  1839.93  1852.40
               25%    1465.50  2160.87  2181.02  2377.40
               50%    1640.02  2324.94  2552.73  2696.15
               75%    1746.70  2477.90  2815.99  2941.84
               max    2135.12  3076.46  3441.11  3593.23
rain_dry_mm    count    38.00    38.00    38.00    38.00
               mean     59.83   125.75   265.40   287.59
               std      18.97    35.85    64.57    72.28
               min      30.19    72.77   159.89   171.91
               25%      46.27    94.59   218.97   239.14
        

In [ ]:
out = sat.merge(bps, on='gaul_name', how='left')
assert len(out) == 152 and out.kabupaten_kota.notna().all()

out = out[['kabupaten_kota', 'jenis', 'tahun',
           'rain_annual_mm', 'rain_dry_mm', 'ndvi_mean', 'ndvi_max']]
out[['rain_annual_mm', 'rain_dry_mm']] = out[['rain_annual_mm', 'rain_dry_mm']].round(2)
out[['ndvi_mean', 'ndvi_max']] = out[['ndvi_mean', 'ndvi_max']].round(4)

out.to_csv('iklim_ndvi_jatim_2019_2022.csv', index=False)
from google.colab import files
files.download('iklim_ndvi_jatim_2019_2022.csv')
print(out.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  kabupaten_kota jenis  tahun  rain_annual_mm  rain_dry_mm  ndvi_mean  \
0           Batu  kota   2019         2135.12        83.88     0.6459   
1         Blitar  kota   2019         1573.31        60.66     0.5455   
2         Kediri  kota   2019         1789.49        47.93     0.5473   
3         Madiun  kota   2019         1783.86        65.10     0.4707   
4         Malang  kota   2019         1696.14        80.46     0.4723   

   ndvi_max  
0    0.8422  
1    0.7239  
2    0.7203  
3    0.6578  
4    0.6391  


In [ ]:
import pandas as pd

viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG').select('avg_rad')
modis = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')
lcover = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1')

def to_df(fc):
    return pd.DataFrame([f['properties'] for f in fc.getInfo()['features']])

# Porsi cropland per kabupaten (WorldCover 2021, kelas 40), tanpa reproject
wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
crop_img = wc.eq(40).rename('crop_share')
crop_df = (to_df(crop_img.reduceRegions(
    collection=regions, reducer=ee.Reducer.mean(), scale=100, tileScale=4))
    .rename(columns={'mean': 'crop_share'}))
print('crop_share selesai')

frames = []
for y in [2019, 2020, 2021, 2022]:
    # Cahaya malam VIIRS: rata-rata radiansi tahunan
    ntl = (viirs.filterDate(f'{y}-01-01', f'{y+1}-01-01')
           .mean().rename('ntl_mean'))
    ntl_df = (to_df(ntl.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=500, tileScale=4))
        .rename(columns={'mean': 'ntl_mean'}))

    # NDVI hanya di lahan pertanian (MODIS land cover: 12 = cropland, 14 = mosaik)
    lc = lcover.filterDate(f'{y}-01-01', f'{y+1}-01-01').first()
    crop_mask = lc.eq(12).Or(lc.eq(14))

    nd = modis.filterDate(f'{y}-01-01', f'{y+1}-01-01')
    ndvi_img = (nd.mean().multiply(0.0001).rename('ndvi_crop_mean')
                .addBands(nd.max().multiply(0.0001).rename('ndvi_crop_max'))
                .updateMask(crop_mask))
    ndvi_df = to_df(ndvi_img.reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(), scale=250, tileScale=4))

    df = (ntl_df.merge(ndvi_df, on='GAUL2_NAME')
                .merge(crop_df, on='GAUL2_NAME'))
    df['tahun'] = y
    frames.append(df)
    print('selesai', y)

sat2 = (pd.concat(frames, ignore_index=True)
        .rename(columns={'GAUL2_NAME': 'gaul_name'}))

print(sat2.shape)
print(sat2.columns.tolist())
print(sat2.isna().sum())
print(sat2.groupby('tahun')[['ntl_mean', 'ndvi_crop_mean', 'ndvi_crop_max', 'crop_share']]
      .describe().T.round(3))

# Cek penamaan: kota besar harus terang dan porsi cropland-nya kecil
chk = (sat2[sat2.tahun == 2022].set_index('gaul_name')
       [['ntl_mean', 'crop_share', 'ndvi_crop_mean']])
print(chk.sort_values('ntl_mean', ascending=False).head(5).round(3))
print(chk.sort_values('crop_share').head(5).round(3))

crop_share selesai
selesai 2019
selesai 2020
selesai 2021
selesai 2022
(152, 6)
['gaul_name', 'ntl_mean', 'ndvi_crop_max', 'ndvi_crop_mean', 'crop_share', 'tahun']
gaul_name         0
ntl_mean          0
ndvi_crop_max     0
ndvi_crop_mean    0
crop_share        0
tahun             0
dtype: int64
tahun                   2019    2020    2021    2022
ntl_mean       count  38.000  38.000  38.000  38.000
               mean    3.550   3.574   2.991   3.014
               std     4.521   4.566   3.517   3.323
               min     0.525   0.567   0.573   0.559
               25%     0.998   1.052   1.018   1.022
               50%     1.452   1.487   1.384   1.421
               75%     4.166   4.309   3.240   3.345
               max    22.479  23.535  18.246  15.174
ndvi_crop_mean count  38.000  38.000  38.000  38.000
               mean    0.593   0.630   0.619   0.631
               std     0.047   0.049   0.048   0.044
               min     0.485   0.525   0.508   0.537
              

In [ ]:
out = sat2.merge(bps, on='gaul_name', how='left')
assert len(out) == 152 and out.kabupaten_kota.notna().all()

out = out[['kabupaten_kota', 'jenis', 'tahun',
           'ntl_mean', 'crop_share', 'ndvi_crop_mean', 'ndvi_crop_max']]
out[['ntl_mean']] = out[['ntl_mean']].round(3)
out[['crop_share', 'ndvi_crop_mean', 'ndvi_crop_max']] = out[['crop_share', 'ndvi_crop_mean', 'ndvi_crop_max']].round(4)

out.to_csv('viirs_cropndvi_jatim_2019_2022.csv', index=False)
from google.colab import files
files.download('viirs_cropndvi_jatim_2019_2022.csv')
print(out.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  kabupaten_kota jenis  tahun  ntl_mean  crop_share  ndvi_crop_mean  \
0           Batu  kota   2019     2.670      0.0652          0.6367   
1         Blitar  kota   2019     6.320      0.2601          0.5995   
2         Kediri  kota   2019     6.128      0.3577          0.6077   
3         Madiun  kota   2019     9.445      0.2687          0.5652   
4         Malang  kota   2019    12.007      0.0924          0.6118   

   ndvi_crop_max  
0         0.8041  
1         0.7863  
2         0.8019  
3         0.8143  
4         0.7969  
